# RTO forecast 

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import statsmodels.api as sm

from datetime import datetime, timedelta

## dataset 

### load forecast data

In [ ]:
offset = 0

today = (pd.Timestamp.today() - pd.Timedelta(days=offset)).normalize()

today_str = (datetime.today() - timedelta(days=offset)).strftime("%m%d") 

load_forecast = pd.read_csv(f'data/load_frcstd_7_day/load_frcstd_7_day_{today_str}.csv', skiprows=[1])

load_forecast['forecast_load_mw'] = pd.to_numeric(load_forecast['forecast_load_mw'], errors='coerce')

In [ ]:
load_forecast = load_forecast[['evaluated_at_datetime_ept', 'forecast_datetime_ending_ept', 'forecast_area', 'forecast_load_mw']].copy()

load_forecast["forecast_datetime_ending_ept"] = pd.to_datetime(load_forecast["forecast_datetime_ending_ept"])

load_forecast = load_forecast[load_forecast["forecast_datetime_ending_ept"] >= today]

In [ ]:
forecast_to_agg = {
    "AE/MIDATL": "AE",
    "AEP": "AEP",
    "AP": "AP",
    "ATSI": "ATSI",
    "BG&E/MIDATL": "BGE",
    "COMED": "COMED",
    "DAYTON": "DAYTON",
    "DEOK": "DEOK",
    "DOMINION": "DOM",
    "DP&L/MIDATL": "DPL",
    "DUQUESNE": "DUQ",
    "EKPC": "EKPC",
    "JCP&L/MIDATL": "JC",
    "METED/MIDATL": "METED",
    "PECO/MIDATL": "PE",
    "PENELEC/MIDATL": "PN",
    "PEPCO/MIDATL": "PEP",
    "PPL/MIDATL": "PL",
    "PSE&G/MIDATL": "PS",
    "RECO/MIDATL": "RE",
    "UGI/MIDATL": "UGI",
    
    # Regions
    "WESTERN_REGION": "WEST_REGION",
    "SOUTHERN_REGION": "SOUTH_REGION",
    "MID_ATLANTIC_REGION": "MIDATL_REGION",

    # Whole system
    "RTO_COMBINED": "RTO"
}

load_forecast["agg_NodeName"] = load_forecast["forecast_area"].map(forecast_to_agg)

In [ ]:
load_forecast = (
    load_forecast
    .sort_values(["forecast_datetime_ending_ept", "agg_NodeName", "evaluated_at_datetime_ept"])
    .drop_duplicates(subset=["forecast_datetime_ending_ept", "agg_NodeName"], keep="last")
    .reset_index(drop=True)
)

In [ ]:
rto_load_forecast = load_forecast[load_forecast['agg_NodeName'] == 'RTO'].copy()

rto_load_forecast['forecast_datetime_ending_ept'] = pd.to_datetime(rto_load_forecast['forecast_datetime_ending_ept'])
rto_load_forecast['hour'] = rto_load_forecast['forecast_datetime_ending_ept'].dt.hour
rto_load_forecast.rename(columns={"agg_NodeName": "zone"}, inplace=True)

In [ ]:
rto_load_forecast.tail()

,evaluated_at_datetime_ept,forecast_datetime_ending_ept,forecast_area,forecast_load_mw,zone,hour
2921,2026-05-01 07:17:11.000,2026-05-07 20:00:00,RTO_COMBINED,87902.0,RTO,20
2946,2026-05-01 07:17:11.000,2026-05-07 21:00:00,RTO_COMBINED,88903.0,RTO,21
2971,2026-05-01 07:17:11.000,2026-05-07 22:00:00,RTO_COMBINED,87427.0,RTO,22
2996,2026-05-01 07:17:11.000,2026-05-07 23:00:00,RTO_COMBINED,83176.0,RTO,23
3021,2026-05-01 07:17:11.000,2026-05-08 00:00:00,RTO_COMBINED,78787.0,RTO,0


### weather data

In [ ]:
run_str = (datetime.today() - timedelta(days=offset)).strftime("%y%m%d")  

hist_files = [
    "data/weather/weather_hist_2023.csv",
    "data/weather/weather_hist_2024.csv",
    "data/weather/weather_hist_2025.csv",
    "data/weather/weather_hist_2026.csv",
]

forecast_file = f"data/weather/weather_forecast_{run_str}.csv"

# --- load historical ---
hist_weather = pd.concat([pd.read_csv(f) for f in hist_files], ignore_index=True)
hist_weather.drop(columns=["weather_code"], errors="ignore", inplace=True)
hist_weather.rename(columns={"time": "datetime_ending_ept"}, inplace=True)
hist_weather["datetime_ending_ept"] = pd.to_datetime(hist_weather["datetime_ending_ept"])

# keep only hist data up to run_str
run_dt = pd.to_datetime(run_str, format="%y%m%d")
hist_weather = hist_weather[hist_weather["datetime_ending_ept"] <= run_dt].copy()

# --- load forecast ---
forecast_weather = pd.read_csv(forecast_file)
forecast_weather.drop(columns=["weather_code"], errors="ignore", inplace=True)
forecast_weather.rename(columns={"time": "datetime_ending_ept"}, inplace=True)
forecast_weather["datetime_ending_ept"] = pd.to_datetime(forecast_weather["datetime_ending_ept"])

# keep only forecast data from forecast_run_str
forecast_weather = forecast_weather[
    forecast_weather["datetime_ending_ept"] > run_dt
].copy()

# dedupe
dedupe_cols = ["zone", "weather_station", "lat", "lon", "datetime_ending_ept"]
hist_weather = hist_weather.drop_duplicates(subset=dedupe_cols, keep="first").reset_index(drop=True)
forecast_weather = forecast_weather.drop_duplicates(subset=dedupe_cols, keep="first").reset_index(drop=True)

# --- fill forecast nulls with hour average ---
forecast_weather["hour"] = forecast_weather["datetime_ending_ept"].dt.hour

weather_vars = [
    'temperature_2m', 
    'apparent_temperature', 
    'dew_point_2m',
    'relative_humidity_2m', 
    'precipitation',
    'rain', 
    'snowfall', 
    'snow_depth', 
    'cloud_cover', 
    'cloud_cover_low',
    'cloud_cover_mid', 
    'cloud_cover_high', 
    'wind_speed_10m',
    'wind_direction_10m', 
    'wind_gusts_10m', 
    'surface_pressure', 
    'pressure_msl',
    'et0_fao_evapotranspiration',
    'vapour_pressure_deficit',
    'shortwave_radiation', 
    'direct_radiation', 
    'diffuse_radiation',
    'global_tilted_irradiance', 
    'direct_normal_irradiance',
    'terrestrial_radiation'
]

# hour average within each zone/station/hour
hourly_avg = (
    forecast_weather.groupby(["zone", "weather_station", "hour"])[weather_vars]
    .mean()
    .reset_index()
)

forecast_weather = forecast_weather.merge(
    hourly_avg,
    on=["zone", "weather_station", "hour"],
    how="left",
    suffixes=("", "_hour_avg")
)

for col in weather_vars:
    forecast_weather[col] = forecast_weather[col].fillna(forecast_weather[f"{col}_hour_avg"])

# drop helper columns
drop_cols = (
    ["hour"]
    + [f"{c}_hour_avg" for c in weather_vars]
)
forecast_weather.drop(columns=drop_cols, inplace=True, errors="ignore")

# --- final zonal_weather ---
zonal_weather = pd.concat([hist_weather, forecast_weather], ignore_index=True)
zonal_weather = zonal_weather.sort_values(
    ["zone", "weather_station", "datetime_ending_ept"]
).reset_index(drop=True)

In [ ]:

for var in weather_vars:
    zonal_weather[f"{var}_weighted"] = (
        zonal_weather[var] * zonal_weather["weather_weight"]
    )

zonal_weather = (
    zonal_weather.groupby(["datetime_ending_ept", "zone"])[
        [f"{v}_weighted" for v in weather_vars]
    ]
    .sum()
    .reset_index()
)

zonal_weather = zonal_weather.rename(columns={
    "temperature_2m_weighted": "temperature_2m",
    "apparent_temperature_weighted": "apparent_temperature",
    "dew_point_2m_weighted": "dew_point_2m",
    "relative_humidity_2m_weighted": "relative_humidity_2m",
    "precipitation_weighted": "precipitation",
    "rain_weighted": "rain",
    "snowfall_weighted": "snowfall",
    "snow_depth_weighted": "snow_depth",
    "cloud_cover_weighted": "cloud_cover",
    "cloud_cover_low_weighted": "cloud_cover_low",
    "cloud_cover_mid_weighted": "cloud_cover_mid",
    "cloud_cover_high_weighted": "cloud_cover_high",
    "wind_speed_10m_weighted": "wind_speed_10m",
    "wind_direction_10m_weighted": "wind_direction_10m",
    "wind_gusts_10m_weighted": "wind_gusts_10m",
    "surface_pressure_weighted": "surface_pressure",
    "pressure_msl_weighted": "pressure_msl",
    "et0_fao_evapotranspiration_weighted": "et0_fao_evapotranspiration",
    "vapour_pressure_deficit_weighted": "vapour_pressure_deficit",
    'shortwave_radiation_weighted': 'shortwave_radiation',
    'direct_radiation_weighted': 'direct_radiation',
    'diffuse_radiation_weighted': 'diffuse_radiation',
    'global_tilted_irradiance_weighted': 'global_tilted_irradiance',
    'direct_normal_irradiance_weighted': 'direct_normal_irradiance',
    'terrestrial_radiation_weighted': 'terrestrial_radiation'
})


In [ ]:
zone_map = {
    "AE": "AE",
    "AEP": "AEP",
    "APS": "AP",
    "ATSI": "ATSI",
    "BGE": "BGE",
    "COMED": "COMED",
    "DAYTON": "DAYTON",
    "DPL": "DPL",
    "DQE": "DUQ",
    "EKPC": "EKPC",
    "JCPL": "JC",
    "METED": "METED",
    "PECO": "PE",
    "PENLC": "PN",
    "PEPCO": "PEP",
    "PL": "PL",
    "PS": "PS",
    "RECO": "RE",
    "UGI": "UGI",
    "VEPCO": "DOM",
    "DUKE": "DEOK"
}

zonal_weather["agg_NodeName"] = zonal_weather["zone"].map(zone_map)

### calendar data

In [ ]:
import holidays

zonal_weather['datetime_ending_ept'] = pd.to_datetime(zonal_weather['datetime_ending_ept'])

zonal_weather['date'] = zonal_weather['datetime_ending_ept'].dt.date
zonal_weather['year'] = zonal_weather['datetime_ending_ept'].dt.year
zonal_weather['month'] = zonal_weather['datetime_ending_ept'].dt.month
zonal_weather['day'] = zonal_weather['datetime_ending_ept'].dt.day
zonal_weather['hour'] = zonal_weather['datetime_ending_ept'].dt.hour
zonal_weather['day_of_week'] = zonal_weather['datetime_ending_ept'].dt.dayofweek

zonal_weather['is_weekend'] = zonal_weather['day_of_week'].isin([5,6]).astype(int)

us_holidays = holidays.US(observed=True)

zonal_weather['is_holiday'] = zonal_weather['datetime_ending_ept'].apply(
    lambda x: int(x.date() in us_holidays)
)

zonal_weather['WkDayBeforeHol'] = zonal_weather['datetime_ending_ept'].apply(
    lambda x: int((x + pd.Timedelta(days=1)) in us_holidays and pd.Timestamp(x).dayofweek < 5)
)

zonal_weather['WkDayAfterHol'] = zonal_weather['datetime_ending_ept'].apply(
    lambda x: int((x - pd.Timedelta(days=1)) in us_holidays and pd.Timestamp(x).dayofweek < 5)
)

from dateutil.easter import easter

def build_event_calendar(start_year, end_year):
    rows = []

    # Federal holidays
    us = holidays.US(years=range(start_year, end_year + 1), observed=True)
    for d, name in us.items():
        rows.append((pd.Timestamp(d), name))

    for y in range(start_year, end_year + 1):
        # Easter
        e = pd.Timestamp(easter(y))
        rows += [
            (e - pd.Timedelta(days=2), "Good Friday"),
            (e, "Easter"),
        ]

        # Fixed-date events 
        rows += [
            (pd.Timestamp(f"{y}-12-24"), "Christmas Eve"),
            (pd.Timestamp(f"{y}-12-31"), "New Year's Eve"),
            (pd.Timestamp(f"{y}-10-31"), "Halloween")
        ]

        # Thanksgiving-based 
        tg = [d for d, n in holidays.US(years=[y]).items() if "Thanksgiving" in n][0]

        rows += [
            (pd.Timestamp(tg) + pd.Timedelta(days=1), "Black Friday"),
            (pd.Timestamp(tg) + pd.Timedelta(days=4), "Cyber Monday"),
        ]

    event_df = pd.DataFrame(rows, columns=["date", "event_name"])

    # combine duplicates
    event_df = (
        event_df.groupby("date")["event_name"]
        .apply(lambda x: ", ".join(sorted(set(x))))
        .reset_index()
        .sort_values("date")
    )

    return event_df

# build calendar
start_year = zonal_weather["datetime_ending_ept"].dt.year.min()
end_year   = zonal_weather["datetime_ending_ept"].dt.year.max()

event_df = build_event_calendar(start_year, end_year)

# prepare join key
zonal_weather["date"] = pd.to_datetime(zonal_weather["datetime_ending_ept"]).dt.normalize()

# merge
zonal_weather = zonal_weather.merge(event_df, on="date", how="left")

# flag
zonal_weather["is_event"] = zonal_weather["event_name"].notna().astype(int)

zonal_weather.drop(columns=["event_name"], inplace=True)



In [ ]:
zonal_wc = zonal_weather.copy()

zonal_wc = zonal_wc[zonal_wc['agg_NodeName'] != 'UGI'].copy()

### load scenario

In [ ]:
# read data
hourly_load = pd.read_csv(f'data/raw_pjm_hrl_load_metered/raw_pjm_hrl_load_metered_{today_str}.csv', skiprows=[1], usecols=lambda c: c not in ["datetime_beginning_utc", "auto_key", "is_verified", "insert_datetime"])

hourly_load["MW"] = pd.to_numeric(hourly_load["MW"], errors='coerce')

In [ ]:
hourly_load["datetime_beginning_ept"] = pd.to_datetime(hourly_load["datetime_beginning_ept"])
hourly_load["datetime_beginning_ept"] += pd.Timedelta(hours=1)

hourly_load = hourly_load.rename(columns={
    "datetime_beginning_ept": "datetime_ending_ept"
})

# desired first columns
first_cols = ["datetime_ending_ept", "zone"]

# reorder dataframe
hourly_load = hourly_load[first_cols + [c for c in hourly_load.columns if c not in first_cols]]

In [ ]:
hourly_rto_load = hourly_load[hourly_load['zone'].isin(['RTO'])].copy()

In [ ]:
hourly_rto_load = (
    hourly_rto_load
    .groupby(['datetime_ending_ept', 'zone'], as_index=False)
    .agg({
        'MW': 'first',
        'nerc_region': 'first',
        'mkt_region': 'first',
    })
)

In [ ]:
hourly_rto_load.tail()

,datetime_ending_ept,zone,MW,nerc_region,mkt_region
29175,2026-04-30 20:00:00,RTO,87140.203,RTO,RTO
29176,2026-04-30 21:00:00,RTO,88523.758,RTO,RTO
29177,2026-04-30 22:00:00,RTO,87450.609,RTO,RTO
29178,2026-04-30 23:00:00,RTO,83482.234,RTO,RTO
29179,2026-05-01 00:00:00,RTO,79299.688,RTO,RTO


In [ ]:
append_load = pd.read_csv(f"data/pjm_All_Instantaneous_Load_rt5/pjm_All_Instantaneous_Load_rt5_{today_str}.csv", skiprows=[1])

append_load["instantaneous_load"] = pd.to_numeric(append_load["instantaneous_load"], errors='coerce')

append_load["datetime_beginning_ept"] = pd.to_datetime(append_load["datetime_beginning_ept"])

In [ ]:
append_load = (
    append_load
    .assign(
        hour_beginning_ept=lambda x: x["datetime_beginning_ept"].dt.floor("h"),
        datetime_ending_ept=lambda x: x["datetime_beginning_ept"].dt.floor("h") + pd.Timedelta(hours=1)
    )
    .groupby(["area", "datetime_ending_ept"], as_index=False)["instantaneous_load"]
    .mean()
    .rename(columns={"instantaneous_load": "load_mw_hourly_avg"})
)

In [ ]:
append_pjm = append_load.loc[append_load["area"] == "PJM RTO"].copy()

max_dt_hourly = hourly_rto_load["datetime_ending_ept"].max()

append_pjm_new = append_pjm.loc[
    append_pjm["datetime_ending_ept"] > max_dt_hourly,
    ["datetime_ending_ept", "load_mw_hourly_avg"]
].copy()

append_pjm_new = append_pjm_new.rename(columns={"load_mw_hourly_avg": "MW"})
append_pjm_new["zone"] = "RTO"
append_pjm_new["nerc_region"] = "RTO"
append_pjm_new["mkt_region"] = "RTO"

hourly_rto_load = pd.concat(
    [hourly_rto_load, append_pjm_new[["datetime_ending_ept", "zone", "MW", "nerc_region", "mkt_region"]]],
    ignore_index=True
).sort_values("datetime_ending_ept").reset_index(drop=True)

In [ ]:
# limit the time range
today_hour = (pd.Timestamp.today() - pd.Timedelta(days=offset)).normalize() + pd.Timedelta(hours=9)

hourly_rto_load = hourly_rto_load[hourly_rto_load['datetime_ending_ept'] <= today_hour].copy()

hourly_rto_load.tail()

,datetime_ending_ept,zone,MW,nerc_region,mkt_region
29232,2026-05-03 05:00:00,RTO,73963.184615,RTO,RTO
29233,2026-05-03 06:00:00,RTO,75466.552462,RTO,RTO
29234,2026-05-03 07:00:00,RTO,77164.801357,RTO,RTO
29235,2026-05-03 08:00:00,RTO,78328.442000,RTO,RTO
29236,2026-05-03 09:00:00,RTO,77898.312692,RTO,RTO


In [ ]:
weather_cols = [
    "temperature_2m",
    "apparent_temperature",
    "dew_point_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "snowfall",
    "snow_depth",
    "cloud_cover",
    "cloud_cover_low",
    "cloud_cover_mid",
    "cloud_cover_high",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
    "surface_pressure",
    "pressure_msl",
    "et0_fao_evapotranspiration",
    "vapour_pressure_deficit",
    'shortwave_radiation', 
    'direct_radiation', 
    'diffuse_radiation',
    'global_tilted_irradiance', 
    'direct_normal_irradiance',
    'terrestrial_radiation'
]


other_cols = [c for c in zonal_wc.columns if c not in weather_cols + ["agg_NodeName"]]

# aggregation dictionary
agg_dict = {}

# mean (keeps original names)
for col in weather_cols:
    agg_dict[col] = ["median", "std"]

# other columns
agg_dict.update({col: "first" for col in other_cols if col != "datetime_ending_ept"})

# groupby
rto_weather = (
    zonal_wc
    .groupby("datetime_ending_ept", as_index=False)
    .agg(agg_dict)
)

# flatten MultiIndex columns
rto_weather.columns = [
    col if isinstance(col, str) else
    f"{col[0]}_std" if col[1] == "std" else col[0]
    for col in rto_weather.columns
]

rto_weather.drop(columns=["zone"], inplace=True)

rto_weather.tail()

,datetime_ending_ept,temperature_2m,temperature_2m_std,apparent_temperature,apparent_temperature_std,dew_point_2m,dew_point_2m_std,relative_humidity_2m,relative_humidity_2m_std,precipitation,...,year,month,day,hour,day_of_week,is_weekend,is_holiday,WkDayBeforeHol,WkDayAfterHol,is_event
29372,2026-05-08 20:00:00,12.465,2.156523,9.94775,2.000393,4.65,2.478725,58.800,15.422265,0.0,...,2026,5,8,20,4,0,0,0,0,0
29373,2026-05-08 21:00:00,11.555,2.143753,9.35155,2.070090,4.65,2.487276,63.175,15.868696,0.0,...,2026,5,8,21,4,0,0,0,0,0
29374,2026-05-08 22:00:00,11.040,2.088980,9.05000,2.106012,4.45,2.558570,67.525,16.536375,0.0,...,2026,5,8,22,4,0,0,0,0,0
29375,2026-05-08 23:00:00,10.665,2.041454,8.63390,2.111745,4.65,2.635054,70.050,17.044022,0.0,...,2026,5,8,23,4,0,0,0,0,0
29376,2026-05-09 00:00:00,10.280,1.999216,8.31910,2.142181,4.60,2.731994,70.625,17.142097,0.0,...,2026,5,9,0,5,1,0,0,0,0


In [ ]:
# merge
rto_load = pd.merge(
    hourly_rto_load,
    rto_weather,
    on="datetime_ending_ept",
    how='right'   
)

rto_load.drop(columns=["mkt_region", "nerc_region"], inplace=True)

# sort just in case
rto_load = rto_load.sort_values('datetime_ending_ept')

rto_load["datetime_ending_ept"] = pd.to_datetime(rto_load["datetime_ending_ept"])

# hour of target timestamp
rto_load["hour"] = rto_load["datetime_ending_ept"].dt.hour


In [ ]:
rto_load.tail()

,datetime_ending_ept,zone,MW,temperature_2m,temperature_2m_std,apparent_temperature,apparent_temperature_std,dew_point_2m,dew_point_2m_std,relative_humidity_2m,...,year,month,day,hour,day_of_week,is_weekend,is_holiday,WkDayBeforeHol,WkDayAfterHol,is_event
29372,2026-05-08 20:00:00,NaN,NaN,12.465,2.156523,9.94775,2.000393,4.65,2.478725,58.800,...,2026,5,8,20,4,0,0,0,0,0
29373,2026-05-08 21:00:00,NaN,NaN,11.555,2.143753,9.35155,2.070090,4.65,2.487276,63.175,...,2026,5,8,21,4,0,0,0,0,0
29374,2026-05-08 22:00:00,NaN,NaN,11.040,2.088980,9.05000,2.106012,4.45,2.558570,67.525,...,2026,5,8,22,4,0,0,0,0,0
29375,2026-05-08 23:00:00,NaN,NaN,10.665,2.041454,8.63390,2.111745,4.65,2.635054,70.050,...,2026,5,8,23,4,0,0,0,0,0
29376,2026-05-09 00:00:00,NaN,NaN,10.280,1.999216,8.31910,2.142181,4.60,2.731994,70.625,...,2026,5,9,0,5,1,0,0,0,0


## RTO Sector Prediction

In [ ]:
hourly_sector_weight = pd.read_csv('data/hrl_sct_wt.csv')

In [ ]:
rto_sector_shares = pd.DataFrame([{"zone": "RTO",  "residential_share": 0.37, "commercial_share": 0.37, "industrial_share": 0.25}])

In [ ]:
df = rto_load_forecast.copy()

df = df.merge(hourly_sector_weight, on="hour", how="left")

df = df.merge(rto_sector_shares, on="zone", how="left")

# unnormalized
df["wtd_res"] = df["residential_share"] * df["w_res"]
df["wtd_com"]  = df["commercial_share"]  * df["w_com"]
df["wtd_ind"]  = df["industrial_share"]  * df["w_ind"]

# normalize
weight_sum = df[["wtd_res", "wtd_com", "wtd_ind"]].sum(axis=1)

df["res_forecast_load_mw"] = df["forecast_load_mw"] * df["wtd_res"] / weight_sum
df["com_forecast_load_mw"]  = df["forecast_load_mw"] * df["wtd_com"]  / weight_sum
df["ind_forecast_load_mw"]  = df["forecast_load_mw"] * df["wtd_ind"]  / weight_sum

rto_load_forecast = df.copy()

In [ ]:
df = rto_load.copy()

df = df.merge(hourly_sector_weight, on="hour", how="left")

df = df.merge(rto_sector_shares, on="zone", how="left")

# unnormalized
df["wtd_res"] = df["residential_share"] * df["w_res"]
df["wtd_com"]  = df["commercial_share"]  * df["w_com"]
df["wtd_ind"]  = df["industrial_share"]  * df["w_ind"]

# normalize
weight_sum = df[["wtd_res", "wtd_com", "wtd_ind"]].sum(axis=1)

df["res_MW"] = df["MW"] * df["wtd_res"] / weight_sum
df["com_MW"]  = df["MW"] * df["wtd_com"]  / weight_sum
df["ind_MW"]  = df["MW"] * df["wtd_ind"]  / weight_sum



In [ ]:
res_rto_load = df[[
    'datetime_ending_ept', 'zone', 'res_MW',
    'temperature_2m', 'apparent_temperature', 'dew_point_2m', 
    'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'snow_depth', 
    'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high',
    'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m',
    'surface_pressure', 'pressure_msl', 'et0_fao_evapotranspiration', 'vapour_pressure_deficit',
    'shortwave_radiation', 'direct_radiation', 'diffuse_radiation', 'global_tilted_irradiance', 'direct_normal_irradiance', 'terrestrial_radiation',
    'temperature_2m_std', 'apparent_temperature_std', 'dew_point_2m_std', 
    'relative_humidity_2m_std', 'precipitation_std', 'rain_std', 'snowfall_std', 'snow_depth_std', 
    'cloud_cover_std', 'cloud_cover_low_std', 'cloud_cover_mid_std', 'cloud_cover_high_std',
    'wind_speed_10m_std', 'wind_direction_10m_std', 'wind_gusts_10m_std',
    'surface_pressure_std', 'pressure_msl_std', 'et0_fao_evapotranspiration_std', 'vapour_pressure_deficit_std',
    'shortwave_radiation_std', 'direct_radiation_std', 'diffuse_radiation_std', 'global_tilted_irradiance_std', 'direct_normal_irradiance_std', 'terrestrial_radiation_std',
    'date', 'year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend', 'is_holiday', 'WkDayBeforeHol', 'WkDayAfterHol','is_event'
]].copy()

com_rto_load = df[[
    'datetime_ending_ept', 'zone', 'com_MW',
    'temperature_2m', 'apparent_temperature', 'dew_point_2m', 
    'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'snow_depth', 
    'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high',
    'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m',
    'surface_pressure', 'pressure_msl', 'et0_fao_evapotranspiration', 'vapour_pressure_deficit',
    'shortwave_radiation', 'direct_radiation', 'diffuse_radiation', 'global_tilted_irradiance', 'direct_normal_irradiance', 'terrestrial_radiation',
    'temperature_2m_std', 'apparent_temperature_std', 'dew_point_2m_std', 
    'relative_humidity_2m_std', 'precipitation_std', 'rain_std', 'snowfall_std', 'snow_depth_std', 
    'cloud_cover_std', 'cloud_cover_low_std', 'cloud_cover_mid_std', 'cloud_cover_high_std',
    'wind_speed_10m_std', 'wind_direction_10m_std', 'wind_gusts_10m_std',
    'surface_pressure_std', 'pressure_msl_std', 'et0_fao_evapotranspiration_std', 'vapour_pressure_deficit_std',
    'shortwave_radiation_std', 'direct_radiation_std', 'diffuse_radiation_std', 'global_tilted_irradiance_std', 'direct_normal_irradiance_std', 'terrestrial_radiation_std',
    'date', 'year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend', 'is_holiday', 'WkDayBeforeHol', 'WkDayAfterHol','is_event'
]].copy()

ind_rto_load = df[[
    'datetime_ending_ept', 'zone', 'ind_MW',
    'temperature_2m', 'apparent_temperature', 'dew_point_2m', 
    'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'snow_depth', 
    'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high',
    'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m',
    'surface_pressure', 'pressure_msl', 'et0_fao_evapotranspiration', 'vapour_pressure_deficit',
    'shortwave_radiation', 'direct_radiation', 'diffuse_radiation', 'global_tilted_irradiance', 'direct_normal_irradiance', 'terrestrial_radiation',
    'temperature_2m_std', 'apparent_temperature_std', 'dew_point_2m_std', 
    'relative_humidity_2m_std', 'precipitation_std', 'rain_std', 'snowfall_std', 'snow_depth_std', 
    'cloud_cover_std', 'cloud_cover_low_std', 'cloud_cover_mid_std', 'cloud_cover_high_std',
    'wind_speed_10m_std', 'wind_direction_10m_std', 'wind_gusts_10m_std',
    'surface_pressure_std', 'pressure_msl_std', 'et0_fao_evapotranspiration_std', 'vapour_pressure_deficit_std',
    'shortwave_radiation_std', 'direct_radiation_std', 'diffuse_radiation_std', 'global_tilted_irradiance_std', 'direct_normal_irradiance_std', 'terrestrial_radiation_std',
    'date', 'year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend', 'is_holiday', 'WkDayBeforeHol', 'WkDayAfterHol','is_event'
]].copy()


### residential prediction

In [ ]:
alert = pd.read_csv("data/emergencymessages/alerts.csv")
alert["start"] = pd.to_datetime(alert["start"].astype(str).str[:-6])
alert["end"]   = pd.to_datetime(alert["end"].astype(str).str[:-6])

In [ ]:
# make sure datetime types match
res_rto_load["datetime_ending_ept"] = pd.to_datetime(res_rto_load["datetime_ending_ept"])

# now your existing code works
res_rto_load["is_alert"] = 0

for _, row in alert.iterrows():
    mask = (
        (res_rto_load["datetime_ending_ept"] >= row["start"]) &
        (res_rto_load["datetime_ending_ept"] <= row["end"])
    )
    res_rto_load.loc[mask, "is_alert"] = 1

In [ ]:
warning = pd.read_csv("data/emergencymessages/warnings.csv")
warning["start"] = pd.to_datetime(warning["start"].astype(str).str[:-6])
warning["end"]   = pd.to_datetime(warning["end"].astype(str).str[:-6])

In [ ]:
res_rto_load["is_warning"] = 0


for _, row in warning.iterrows():
    mask = (
        (res_rto_load["datetime_ending_ept"] >= row["start"]) &
        (res_rto_load["datetime_ending_ept"] <= row["end"])
    )
    res_rto_load.loc[mask, "is_warning"] = 1


In [ ]:
action = pd.read_csv("data/emergencymessages/actions.csv")
action["start"] = pd.to_datetime(action["start"].astype(str).str[:-6])
action["end"]   = pd.to_datetime(action["end"].astype(str).str[:-6])


In [ ]:
res_rto_load["is_action"] = 0

for _, row in action.iterrows():
    mask = (
        (res_rto_load["datetime_ending_ept"] >= row["start"]) &
        (res_rto_load["datetime_ending_ept"] <= row["end"])
    )
    res_rto_load.loc[mask, "is_action"] = 1

In [ ]:
daily_temp = (
    res_rto_load.groupby("date", as_index=False)["temperature_2m"]
    .agg(temp_min="min", temp_max="max")
)

res_rto_load = res_rto_load.merge(daily_temp, on="date", how="left")

res_rto_load["temp_f"] = res_rto_load["temperature_2m"] * 9/5 + 32

base_temp = 65

res_rto_load["HDD"] = np.maximum(base_temp - res_rto_load["temp_f"], 0)
res_rto_load["CDD"] = np.maximum(res_rto_load["temp_f"] - base_temp, 0)

res_rto_load["HDD_wind"] = res_rto_load["HDD"] * res_rto_load["wind_speed_10m"]
res_rto_load["CDD_cloud"] = res_rto_load["CDD"] * (1 - res_rto_load["cloud_cover"] / 100)

res_rto_load["wind_dir_10m_sin"] = np.sin(np.deg2rad(res_rto_load["wind_direction_10m"]))
res_rto_load["wind_dir_10m_cos"] = np.cos(np.deg2rad(res_rto_load["wind_direction_10m"]))

res_rto_load["wind_chill"] = (
    35.74
    + 0.6215 * res_rto_load["temp_f"]
    - 35.75 * (res_rto_load["wind_speed_10m"] ** 0.16)
    + 0.4275 * res_rto_load["temp_f"] * (res_rto_load["wind_speed_10m"] ** 0.16)
)

RH = res_rto_load["relative_humidity_2m"]
T = res_rto_load["temp_f"]

res_rto_load["heat_index_f"] = (
    -42.379
    + 2.04901523 * T
    + 10.14333127 * RH
    - 0.22475541 * T * RH
    - 0.00683783 * T**2
    - 0.05481717 * RH**2
    + 0.00122874 * T**2 * RH
    + 0.00085282 * T * RH**2
    - 0.00000199 * T**2 * RH**2
)

res_rto_load["feels_like_temp"] = np.where(
    (res_rto_load["temp_f"] <= 50) & (res_rto_load["wind_speed_10m"] > 3),
    res_rto_load["wind_chill"],
    np.where(
        (res_rto_load["temp_f"] >= 80) & (RH >= 40),
        res_rto_load["heat_index_f"],
        res_rto_load["temp_f"]
    )
)

# hour-to-hour absolute temperature change
res_rto_load["temp_diff_1h_abs"] = res_rto_load["temperature_2m"].diff().abs()

# cumulative total variation over past 6 hours
res_rto_load["temp_total_variation_6h"] = (
    res_rto_load["temp_diff_1h_abs"].rolling(6).sum()
)

# cumulative total variation over past 12 hours
res_rto_load["temp_total_variation_12h"] = (
    res_rto_load["temp_diff_1h_abs"].rolling(12).sum()
)

# cumulative total variation over past 24 hours
res_rto_load["temp_total_variation_24h"] = (
    res_rto_load["temp_diff_1h_abs"].rolling(24).sum()
)

res_rto_load["temp_fcst_1h"] = res_rto_load["temperature_2m"].shift(-1)

horizon = 6

res_rto_load["temp_lead_mean_6h"] = sum(
    res_rto_load["temperature_2m"].shift(-i)
    for i in range(1, horizon + 1)
) / horizon


res_rto_load.drop(columns=["wind_chill", "temp_diff_1h_abs", "heat_index_f"], inplace=True)



In [ ]:
res_rto_load = res_rto_load.sort_values("datetime_ending_ept").copy()
res_rto_load["datetime_ending_ept"] = pd.to_datetime(res_rto_load["datetime_ending_ept"])

# hour of target timestamp
res_rto_load["hour"] = res_rto_load["datetime_ending_ept"].dt.hour

# if target hour < 10, use 1 day back
# if target hour >= 10, use 2 days back
res_rto_load["day_shift"] = np.where(res_rto_load["hour"].isin(range(1,10)), 1, 2)

# lookup index from historical MW
hist = (
    res_rto_load[["datetime_ending_ept", "res_MW"]]
    .drop_duplicates("datetime_ending_ept")
    .sort_values("datetime_ending_ept")
    .set_index("datetime_ending_ept")
)

# choose lag hours to create
lag_hours = [1, 2, 3, 6, 12, 24, 36, 48, 72, 96, 120, 144, 168]

# build lag lookup timestamps + features
helper_cols = []
feature_cols = []

for lag in lag_hours:
    time_col = f"lag{lag}_time"
    feat_col = f"MW_lag_{lag}"

    res_rto_load[time_col] = (
        res_rto_load["datetime_ending_ept"]
        - pd.to_timedelta(res_rto_load["day_shift"], unit="D")
        - pd.Timedelta(hours=lag)
    )

    res_rto_load[feat_col] = hist["res_MW"].reindex(res_rto_load[time_col]).to_numpy()

    helper_cols.append(time_col)
    feature_cols.append(feat_col)

# rolling stats based only on historical series
hist["MW_roll_24_base"] = hist["res_MW"].shift(1).rolling(24).mean()
hist["MW_roll_48_base"] = hist["res_MW"].shift(1).rolling(48).mean()
hist["MW_roll_72_base"] = hist["res_MW"].shift(1).rolling(72).mean()
hist["MW_roll_168_base"] = hist["res_MW"].shift(1).rolling(168).mean()

hist["MW_std_24_base"] = hist["res_MW"].shift(1).rolling(24).std()
hist["MW_std_168_base"] = hist["res_MW"].shift(1).rolling(168).std()

# rolling lookup time
res_rto_load["roll_time"] = (
    res_rto_load["datetime_ending_ept"]
    - pd.to_timedelta(res_rto_load["day_shift"], unit="D")
)

res_rto_load["MW_roll_24"] = hist["MW_roll_24_base"].reindex(res_rto_load["roll_time"]).to_numpy()
res_rto_load["MW_roll_48"] = hist["MW_roll_48_base"].reindex(res_rto_load["roll_time"]).to_numpy()
res_rto_load["MW_roll_72"] = hist["MW_roll_72_base"].reindex(res_rto_load["roll_time"]).to_numpy()
res_rto_load["MW_roll_168"] = hist["MW_roll_168_base"].reindex(res_rto_load["roll_time"]).to_numpy()

res_rto_load["MW_std_24"] = hist["MW_std_24_base"].reindex(res_rto_load["roll_time"]).to_numpy()
res_rto_load["MW_std_168"] = hist["MW_std_168_base"].reindex(res_rto_load["roll_time"]).to_numpy()

feature_cols += [
    "MW_roll_24", "MW_roll_48", "MW_roll_72", "MW_roll_168",
    "MW_std_24", "MW_std_168"
]

# drop rows without enough history
res_rto_load = res_rto_load.dropna(subset=feature_cols).copy()

# optional: remove helper columns
res_rto_load.drop(columns=["day_shift", "roll_time"] + helper_cols, inplace=True)

In [ ]:
res_rto_load.tail()

,datetime_ending_ept,zone,res_MW,temperature_2m,apparent_temperature,dew_point_2m,relative_humidity_2m,precipitation,rain,snowfall,...,MW_lag_96,MW_lag_120,MW_lag_144,MW_lag_168,MW_roll_24,MW_roll_48,MW_roll_72,MW_roll_168,MW_std_24,MW_std_168
29277,2026-05-04 21:00:00,NaN,NaN,15.9250,13.8750,9.325,74.895,0.0000,0.0,0.0,...,39568.454801,39753.818046,37290.707054,37051.663298,29564.214641,30301.938640,30575.679534,30612.436944,3977.159594,4495.017901
29278,2026-05-04 22:00:00,NaN,NaN,15.0340,13.2125,9.385,76.380,0.0000,0.0,0.0,...,37265.812760,37137.361583,35409.086799,35018.549499,29501.494684,30253.945952,30538.766906,30609.318806,3847.555761,4490.703965
29279,2026-05-04 23:00:00,NaN,NaN,15.1000,12.9000,9.405,72.910,0.0000,0.0,0.0,...,33041.639777,32811.354407,31667.139818,31498.677030,29445.877141,30215.313725,30512.777553,30609.716301,3751.648554,4491.099524
29280,2026-05-05 00:00:00,NaN,NaN,14.6625,12.5500,9.625,73.750,0.0125,0.0,0.0,...,29099.221255,28918.152386,28154.854290,28143.129692,29406.864209,30193.464643,30500.516287,30612.467418,3718.924077,4491.788845
29290,2026-05-05 10:00:00,NaN,NaN,19.8225,17.3000,10.385,57.750,0.0000,0.0,0.0,...,30361.793233,30427.136332,29790.105529,27702.512790,29430.745960,29916.634644,30402.019714,30702.459356,3672.605956,4416.657556


In [ ]:
rto_load_forecast.rename(columns={"forecast_datetime_ending_ept": "datetime_ending_ept"}, inplace=True)

rto_load_forecast["datetime_ending_ept"] = pd.to_datetime(
        rto_load_forecast["datetime_ending_ept"]
    )

res_rto_load = pd.merge(
    res_rto_load,
    rto_load_forecast[["datetime_ending_ept", "res_forecast_load_mw"]],
    on="datetime_ending_ept",
    how="left"
)

# res_rto_load.dropna(subset=["temperature_2m"], inplace=True)

In [ ]:
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

results = []
feat_imp_results = []

start_date = (pd.Timestamp.today() - pd.Timedelta(days=offset)).normalize() # pd.Timestamp("2026-03-25")
end_date   = start_date + pd.Timedelta(days=2) - pd.Timedelta(hours=1) # pd.Timestamp("2026-03-26")

current_date = start_date

while current_date <= end_date:

    print(f"Running for {current_date.date()}")

    # Define cutoff (day before 10:00)
    cutoff = (current_date - pd.Timedelta(days=1)).replace(hour=10)

    # Train set
    train_df = res_rto_load[
        res_rto_load["datetime_ending_ept"] <= cutoff
    ].copy()

    # Test set (full target day)
    test_start = current_date.replace(hour=1)
    test_end   = current_date.replace(hour=23) + pd.Timedelta(hours=1)

    test_df = res_rto_load[
        (res_rto_load["datetime_ending_ept"] >= test_start) &
        (res_rto_load["datetime_ending_ept"] <= test_end)
    ].copy()

    # Target / features
    target = "res_MW"
    drop_cols = ["res_MW", "datetime_ending_ept", "zone", "date", "res_forecast_load_mw"]
    features = [col for col in train_df.columns if col not in drop_cols]

    # Time-based validation split
    valid_hours = 24 * 14
    train_part = train_df.iloc[:-valid_hours].copy()
    valid_part = train_df.iloc[-valid_hours:].copy()

    X_train = train_part[features]
    y_train = train_part[target]

    X_valid = valid_part[features]
    y_valid = valid_part[target]

    X_test = test_df[features]

    # Build weights based on PJM forecast miss > 1%
    train_dev_pct = (
        (train_part["res_MW"] - train_part["res_forecast_load_mw"]).abs()
        / train_part["res_forecast_load_mw"].abs().clip(lower=1e-6)
    )
    valid_dev_pct = (
        (valid_part["res_MW"] - valid_part["res_forecast_load_mw"]).abs()
        / valid_part["res_forecast_load_mw"].abs().clip(lower=1e-6)
    )

    train_extreme = train_dev_pct > 0.02
    valid_extreme = valid_dev_pct > 0.02

    # heavier penalty on extreme periods
    train_weights = np.where(train_extreme, 2, 1)
    valid_weights = np.where(valid_extreme, 2, 1)

    # Custom eval metric: RMSE only on extreme periods
    valid_forecast = valid_part["res_forecast_load_mw"].to_numpy()

    def extreme_rmse_eval(y_true, y_pred):
        extreme_mask = (
            np.abs(y_true - valid_forecast) /
            np.clip(np.abs(valid_forecast), 1e-6, None)
        ) > 0.02

        if extreme_mask.sum() == 0:
            return ("extreme_rmse", 0.0, False)

        rmse = np.sqrt(np.mean((y_true[extreme_mask] - y_pred[extreme_mask]) ** 2))
        return ("extreme_rmse", rmse, False)

    # Train model
    model = LGBMRegressor(
        objective="regression",
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=64,
        max_depth=-1,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42
    )

    model.fit(
        X_train,
        y_train,
        sample_weight=train_weights,
        eval_set=[(X_valid, y_valid)],
        eval_sample_weight=[valid_weights],
        eval_metric=lambda yt, yp: [
            extreme_rmse_eval(yt, yp)
        ],
        callbacks=[
            lgb.early_stopping(100, first_metric_only=True),
            lgb.log_evaluation(100)
        ]
    )

    # Feature importance
    feat_imp = pd.DataFrame({
        "date": current_date,
        "feature": X_train.columns,
        "importance": model.feature_importances_
    }).sort_values(by="importance", ascending=False)

    feat_imp["rank"] = range(1, len(feat_imp) + 1)
    feat_imp_results.append(feat_imp)

    # Predict
    test_df["res_MW_pred"] = model.predict(X_test)

    # Metrics on test day
    mae = mean_absolute_error(test_df["res_forecast_load_mw"], test_df["res_MW_pred"])
    rmse = np.sqrt(mean_squared_error(test_df["res_forecast_load_mw"], test_df["res_MW_pred"]))

    print(f"MAE: {mae:.2f}, RMSE: {rmse:.2f}")

    test_df["date"] = current_date
    test_df["MAE"] = mae
    test_df["RMSE"] = rmse

    results.append(test_df)

    current_date += pd.Timedelta(days=1)

# combine all days
res_final = pd.concat(results).reset_index(drop=True)

Running for 2026-05-03
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005730 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21325
[LightGBM] [Info] Number of data points in the train set: 28003, number of used features: 97
[LightGBM] [Info] Start training from score 34948.059534
Training until validation scores don't improve for 100 rounds
[100]	valid_0's l2: 1.32502e+06	valid_0's extreme_rmse: 0
[200]	valid_0's l2: 1.14484e+06	valid_0's extreme_rmse: 0
[300]	valid_0's l2: 1.07783e+06	valid_0's extreme_rmse: 0
[400]	valid_0's l2: 1.02225e+06	valid_0's extreme_rmse: 0
[500]	valid_0's l2: 994637	valid_0's extreme_rmse: 0
[600]	valid_0's l2: 963642	valid_0's extreme_rmse: 0
[700]	valid_0's l2: 940908	valid_0's extreme_rmse: 0
[800]	valid_0's l2: 929494	valid_0's extreme_rmse: 0
[900]	valid_0's l2: 920524	valid_0's extreme_rmse: 0
[1000]	valid_0's l2: 912938	valid_0's extreme_rmse: 0
Did not meet earl

In [ ]:
mean_mae = res_final["MAE"].mean()
mean_rmse = res_final["RMSE"].mean()
print(mean_mae, mean_rmse)

748.6919250421291 862.2745812925009


In [ ]:
import plotly.graph_objects as go

plot_df = res_final.sort_values("datetime_ending_ept")

fig = go.Figure()


fig.add_trace(go.Scatter(
    x=plot_df["datetime_ending_ept"],
    y=plot_df["res_MW_pred"],
    mode="lines",
    name="Model Prediction (MW_pred)"
))

fig.add_trace(go.Scatter(
    x=plot_df["datetime_ending_ept"],
    y=plot_df["res_forecast_load_mw"],
    mode="lines",
    name="PJM Forecast (forecast_load_mw)"
))

fig.update_layout(
    title=(
        f"Model vs PJM Forecast Load "
        f"({start_date.strftime('%Y-%m-%d')} - {end_date.strftime('%Y-%m-%d')})<br>"
        f"Model → MAE: {mean_mae:.2f}, RMSE: {mean_rmse:.2f} "),
    xaxis_title="Time",
    yaxis_title="MW",
    template="plotly_white"
)

fig.show()

### commercial prediction

In [ ]:
# make sure datetime types match
com_rto_load["datetime_ending_ept"] = pd.to_datetime(com_rto_load["datetime_ending_ept"])

# alert 
com_rto_load["is_alert"] = 0

for _, row in alert.iterrows():
    mask = (
        (com_rto_load["datetime_ending_ept"] >= row["start"]) &
        (com_rto_load["datetime_ending_ept"] <= row["end"])
    )
    com_rto_load.loc[mask, "is_alert"] = 1

com_rto_load["is_warning"] = 0

for _, row in warning.iterrows():
    mask = (
        (com_rto_load["datetime_ending_ept"] >= row["start"]) &
        (com_rto_load["datetime_ending_ept"] <= row["end"])
    )
    com_rto_load.loc[mask, "is_warning"] = 1

com_rto_load["is_action"] = 0

for _, row in action.iterrows():
    mask = (
        (com_rto_load["datetime_ending_ept"] >= row["start"]) &
        (com_rto_load["datetime_ending_ept"] <= row["end"])
    )
    com_rto_load.loc[mask, "is_action"] = 1


# temp
daily_temp = (
    com_rto_load.groupby("date", as_index=False)["temperature_2m"]
    .agg(temp_min="min", temp_max="max")
)

com_rto_load = com_rto_load.merge(daily_temp, on="date", how="left")

com_rto_load["temp_f"] = com_rto_load["temperature_2m"] * 9/5 + 32

base_temp = 65

com_rto_load["HDD"] = np.maximum(base_temp - com_rto_load["temp_f"], 0)
com_rto_load["CDD"] = np.maximum(com_rto_load["temp_f"] - base_temp, 0)

com_rto_load["wind_chill"] = (
    35.74
    + 0.6215 * com_rto_load["temp_f"]
    - 35.75 * (com_rto_load["wind_speed_10m"] ** 0.16)
    + 0.4275 * com_rto_load["temp_f"] * (com_rto_load["wind_speed_10m"] ** 0.16)
)

RH = com_rto_load["relative_humidity_2m"]
T = com_rto_load["temp_f"]

com_rto_load["heat_index_f"] = (
    -42.379
    + 2.04901523 * T
    + 10.14333127 * RH
    - 0.22475541 * T * RH
    - 0.00683783 * T**2
    - 0.05481717 * RH**2
    + 0.00122874 * T**2 * RH
    + 0.00085282 * T * RH**2
    - 0.00000199 * T**2 * RH**2
)

com_rto_load["feels_like_temp"] = np.where(
    (com_rto_load["temp_f"] <= 50) & (com_rto_load["wind_speed_10m"] > 3),
    com_rto_load["wind_chill"],
    np.where(
        (com_rto_load["temp_f"] >= 80) & (RH >= 40),
        com_rto_load["heat_index_f"],
        com_rto_load["temp_f"]
    )
)

com_rto_load["wind_dir_10m_sin"] = np.sin(np.deg2rad(com_rto_load["wind_direction_10m"]))
com_rto_load["wind_dir_10m_cos"] = np.cos(np.deg2rad(com_rto_load["wind_direction_10m"]))

com_rto_load["HDD_wind"] = com_rto_load["HDD"] * com_rto_load["wind_speed_10m"]
com_rto_load["CDD_cloud"] = com_rto_load["CDD"] * (1 - com_rto_load["cloud_cover"] / 100)

# hour-to-hour absolute temperature change
com_rto_load["temp_diff_1h_abs"] = com_rto_load["temperature_2m"].diff().abs()

# cumulative total variation over past 6 hours
com_rto_load["temp_total_variation_6h"] = (
    com_rto_load["temp_diff_1h_abs"].rolling(6).sum()
)

# cumulative total variation over past 12 hours
com_rto_load["temp_total_variation_12h"] = (
    com_rto_load["temp_diff_1h_abs"].rolling(12).sum()
)

# cumulative total variation over past 24 hours
com_rto_load["temp_total_variation_24h"] = (
    com_rto_load["temp_diff_1h_abs"].rolling(24).sum()
)

com_rto_load.drop(columns=["wind_chill", "temp_diff_1h_abs", "heat_index_f"], inplace=True)

# if target hour < 10, use 1 day back
# if target hour >= 10, use 2 days back
com_rto_load["day_shift"] = np.where(com_rto_load["hour"].isin(range(1, 10)), 1, 2)

# lookup index
hist = (
    com_rto_load[["datetime_ending_ept", "com_MW"]]
    .drop_duplicates("datetime_ending_ept")
    .sort_values("datetime_ending_ept")
    .set_index("datetime_ending_ept")
)

# choose lag hours to create
lag_hours = [1, 2, 3, 6, 12, 24, 36, 48, 72, 96, 120, 144, 168]

# build lag lookup timestamps + features
helper_cols = []
feature_cols = []

for lag in lag_hours:
    time_col = f"lag{lag}_time"
    feat_col = f"MW_lag_{lag}"

    com_rto_load[time_col] = (
        com_rto_load["datetime_ending_ept"]
        - pd.to_timedelta(com_rto_load["day_shift"], unit="D")
        - pd.Timedelta(hours=lag)
    )

    com_rto_load[feat_col] = hist["com_MW"].reindex(com_rto_load[time_col]).to_numpy()

    helper_cols.append(time_col)
    feature_cols.append(feat_col)

# rolling stats based only on historical series
hist["MW_roll_24_base"] = hist["com_MW"].shift(1).rolling(24).mean()
hist["MW_roll_48_base"] = hist["com_MW"].shift(1).rolling(48).mean()
hist["MW_roll_72_base"] = hist["com_MW"].shift(1).rolling(72).mean()
hist["MW_roll_168_base"] = hist["com_MW"].shift(1).rolling(168).mean()

hist["MW_std_24_base"] = hist["com_MW"].shift(1).rolling(24).std()
hist["MW_std_168_base"] = hist["com_MW"].shift(1).rolling(168).std()

# rolling lookup time
com_rto_load["roll_time"] = (
    com_rto_load["datetime_ending_ept"]
    - pd.to_timedelta(com_rto_load["day_shift"], unit="D")
)

com_rto_load["MW_roll_24"] = hist["MW_roll_24_base"].reindex(com_rto_load["roll_time"]).to_numpy()
com_rto_load["MW_roll_48"] = hist["MW_roll_48_base"].reindex(com_rto_load["roll_time"]).to_numpy()
com_rto_load["MW_roll_72"] = hist["MW_roll_72_base"].reindex(com_rto_load["roll_time"]).to_numpy()
com_rto_load["MW_roll_168"] = hist["MW_roll_168_base"].reindex(com_rto_load["roll_time"]).to_numpy()

com_rto_load["MW_std_24"] = hist["MW_std_24_base"].reindex(com_rto_load["roll_time"]).to_numpy()
com_rto_load["MW_std_168"] = hist["MW_std_168_base"].reindex(com_rto_load["roll_time"]).to_numpy()

feature_cols += [
    "MW_roll_24", "MW_roll_48", "MW_roll_72", "MW_roll_168",
    "MW_std_24", "MW_std_168"
]

# drop rows without enough history
com_rto_load = com_rto_load.dropna(subset=feature_cols).copy()

# optional: remove helper columns
com_rto_load.drop(columns=["day_shift", "roll_time"] + helper_cols, inplace=True)

In [ ]:
com_rto_load.tail()

,datetime_ending_ept,zone,com_MW,temperature_2m,apparent_temperature,dew_point_2m,relative_humidity_2m,precipitation,rain,snowfall,...,MW_lag_144,MW_lag_168,MW_roll_24,MW_roll_48,MW_roll_72,MW_roll_168,MW_std_24,MW_std_168,temp_min_y,temp_max_y
28397,2026-05-04 21:00:00,NaN,NaN,15.9250,13.8750,9.325,74.895,0.0000,0.0,0.0,...,30160.661498,29967.323308,29002.811966,29715.550071,29979.669225,29997.642120,1197.070510,1787.942404,8.9000,20.4250
28398,2026-05-04 22:00:00,NaN,NaN,15.0340,13.2125,9.385,76.380,0.0000,0.0,0.0,...,30375.852975,30040.828700,28952.084169,29676.733665,29949.814353,29995.120176,1143.937358,1788.284204,8.9000,20.4250
28399,2026-05-04 23:00:00,NaN,NaN,15.1000,12.9000,9.405,72.910,0.0000,0.0,0.0,...,30114.570918,29954.367483,28904.372397,29643.592827,29927.519264,29995.461168,1065.408325,1788.298434,8.9000,20.4250
28400,2026-05-05 00:00:00,NaN,NaN,14.6625,12.5500,9.625,73.750,0.0125,0.0,0.0,...,29521.720457,29509.426651,28867.272181,29622.814955,29915.859140,29998.077405,991.896942,1788.559445,14.6625,19.8225
28401,2026-05-05 10:00:00,NaN,NaN,19.8225,17.3000,10.385,57.750,0.0000,0.0,0.0,...,31019.159342,28845.438550,28892.444989,29355.318651,29825.804841,30089.836024,845.884081,1647.387725,14.6625,19.8225


In [ ]:
rto_load_forecast.tail()

,evaluated_at_datetime_ept,datetime_ending_ept,forecast_area,forecast_load_mw,zone,hour,w_res,w_com,w_ind,residential_share,commercial_share,industrial_share,wtd_res,wtd_com,wtd_ind,res_forecast_load_mw,com_forecast_load_mw,ind_forecast_load_mw
116,2026-05-01 07:17:11.000,2026-05-07 20:00:00,RTO_COMBINED,87902.0,RTO,20,1.318710,1.044747,0.883513,0.37,0.37,0.25,0.487923,0.386557,0.220878,39155.597065,31021.014401,17725.388535
117,2026-05-01 07:17:11.000,2026-05-07 21:00:00,RTO_COMBINED,88903.0,RTO,21,1.275895,1.031941,0.889199,0.37,0.37,0.25,0.472081,0.381818,0.222300,38997.821869,31541.373105,18363.805026
118,2026-05-01 07:17:11.000,2026-05-07 22:00:00,RTO_COMBINED,87427.0,RTO,22,1.189888,1.020751,0.897309,0.37,0.37,0.25,0.440259,0.377678,0.224327,36929.702915,31680.320718,18816.976366
119,2026-05-01 07:17:11.000,2026-05-07 23:00:00,RTO_COMBINED,83176.0,RTO,23,1.057430,1.005586,0.904654,0.37,0.37,0.25,0.391249,0.372067,0.226164,32888.533090,31276.082014,19011.384896
120,2026-05-01 07:17:11.000,2026-05-08 00:00:00,RTO_COMBINED,78787.0,RTO,0,0.934257,0.979614,0.909087,0.37,0.37,0.25,0.345675,0.362457,0.227272,29115.452472,30528.953906,19142.593622


In [ ]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

results = []
feat_imp_results = []

start_date = (pd.Timestamp.today() - pd.Timedelta(days=offset)).normalize() # pd.Timestamp("2026-03-25")
end_date   = start_date + pd.Timedelta(days=2) - pd.Timedelta(hours=1) # pd.Timestamp("2026-03-26")

current_date = start_date

while current_date <= end_date:

    print(f"Running for {current_date.date()}")

    # Define cutoff (day before 10:00)
    cutoff = (current_date - pd.Timedelta(days=1)).replace(hour=10)

    # Train set
    train_df = com_rto_load[
        com_rto_load["datetime_ending_ept"] <= cutoff
    ].copy()

    # Test set (full target day)
    test_start = current_date.replace(hour=1)
    test_end   = current_date.replace(hour=23) + pd.Timedelta(hours=1)

    test_df = com_rto_load[
        (com_rto_load["datetime_ending_ept"] >= test_start) &
        (com_rto_load["datetime_ending_ept"] <= test_end)
    ].copy()

    # Features
    target = "com_MW"

    drop_cols = ["com_MW", "datetime_ending_ept", "zone", "date"]
    features = [col for col in train_df.columns if col not in drop_cols]

    # Time-based validation split from training data
    # use the most recent 14 days of hourly data as validation
    valid_hours = 24 * 14
    
    train_part = train_df.iloc[:-valid_hours].copy()
    valid_part = train_df.iloc[-valid_hours:].copy()

    X_train = train_part[features]
    y_train = train_part[target]

    X_valid = valid_part[features]
    y_valid = valid_part[target]

    X_test = test_df[features]

    # Train model with recommended params + early stopping
    model = LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=64,
        max_depth=-1,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="rmse",
        callbacks=[
            __import__("lightgbm").early_stopping(100),
            __import__("lightgbm").log_evaluation(100)
        ]
    )

    # feature importance for this day 
    feat_imp = pd.DataFrame({
        "date": current_date,
        "feature": X_train.columns,
        "importance": model.feature_importances_
    }).sort_values(by="importance", ascending=False)

    feat_imp["rank"] = range(1, len(feat_imp) + 1)
    feat_imp_results.append(feat_imp)


    # Predict
    test_df["com_MW_pred"] = model.predict(X_test)

    # Merge with PJM forecast
    forecast_df = rto_load_forecast[
        (rto_load_forecast["datetime_ending_ept"] >= test_start) &
        (rto_load_forecast["datetime_ending_ept"] <= test_end)
    ].copy()

    compare_df = pd.merge(
        test_df,
        forecast_df[["datetime_ending_ept", "com_forecast_load_mw"]],
        on="datetime_ending_ept",
        how="left"
    )

    # Metrics
    mae = mean_absolute_error(compare_df["com_forecast_load_mw"], compare_df["com_MW_pred"])
    rmse = np.sqrt(mean_squared_error(compare_df["com_forecast_load_mw"], compare_df["com_MW_pred"]))

    print(f"MAE: {mae:.2f}, RMSE: {rmse:.2f}")

    compare_df["date"] = current_date
    compare_df["MAE"] = mae
    compare_df["RMSE"] = rmse

    results.append(compare_df)

    # move to next day
    current_date += pd.Timedelta(days=1)

# combine all days
com_final = pd.concat(results).reset_index(drop=True)



Running for 2026-05-03
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005423 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21325
[LightGBM] [Info] Number of data points in the train set: 26959, number of used features: 97
[LightGBM] [Info] Start training from score 34263.784674
Training until validation scores don't improve for 100 rounds
[100]	valid_0's rmse: 970.568	valid_0's l2: 942003
[200]	valid_0's rmse: 940.129	valid_0's l2: 883842
[300]	valid_0's rmse: 881.37	valid_0's l2: 776813
[400]	valid_0's rmse: 875.27	valid_0's l2: 766097
[500]	valid_0's rmse: 862.534	valid_0's l2: 743964
[600]	valid_0's rmse: 856.426	valid_0's l2: 733466
[700]	valid_0's rmse: 851.443	valid_0's l2: 724955
[800]	valid_0's rmse: 845.424	valid_0's l2: 714741
[900]	valid_0's rmse: 843.191	valid_0's l2: 710971
[1000]	valid_0's rmse: 841.221	valid_0's l2: 707652
Did not meet early stopping. Best iteration is:
[993]	valid

In [ ]:
mean_mae = com_final["MAE"].mean()
mean_rmse = com_final["RMSE"].mean()
print(mean_mae, mean_rmse)

557.7752573111171 640.5840397975768


In [ ]:
import plotly.graph_objects as go

plot_df = com_final.sort_values("datetime_ending_ept")

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=plot_df["datetime_ending_ept"],
    y=plot_df["com_MW_pred"],
    mode="lines",
    name="Model Prediction (MW_pred)"
))

fig.add_trace(go.Scatter(
    x=plot_df["datetime_ending_ept"],
    y=plot_df["com_forecast_load_mw"],
    mode="lines",
    name="PJM Forecast (forecast_load_mw)"
))

fig.update_layout(
    title=(
        f"Model vs PJM Forecast Load "
        f"({start_date.strftime('%Y-%m-%d')} - {end_date.strftime('%Y-%m-%d')})<br>"
        f"Model → MAE: {mean_mae:.2f}, RMSE: {mean_rmse:.2f} "
    ),
    xaxis_title="Time",
    yaxis_title="MW",
    template="plotly_white"
)

fig.show()

### industrial prediction

In [ ]:
ind_rto_load["datetime_ending_ept"] = pd.to_datetime(ind_rto_load["datetime_ending_ept"])

ind_rto_load["is_alert"] = 0

for _, row in alert.iterrows():
    mask = (
        (ind_rto_load["datetime_ending_ept"] >= row["start"]) &
        (ind_rto_load["datetime_ending_ept"] <= row["end"])
    )
    ind_rto_load.loc[mask, "is_alert"] = 1

ind_rto_load["is_warning"] = 0

for _, row in warning.iterrows():
    mask = (
        (ind_rto_load["datetime_ending_ept"] >= row["start"]) &
        (ind_rto_load["datetime_ending_ept"] <= row["end"])
    )
    ind_rto_load.loc[mask, "is_warning"] = 1

ind_rto_load["is_action"] = 0

for _, row in action.iterrows():
    mask = (
        (ind_rto_load["datetime_ending_ept"] >= row["start"]) &
        (ind_rto_load["datetime_ending_ept"] <= row["end"])
    )
    ind_rto_load.loc[mask, "is_action"] = 1

# temp
daily_temp = (
    ind_rto_load.groupby("date", as_index=False)["temperature_2m"]
    .agg(temp_min="min", temp_max="max")
)

ind_rto_load = ind_rto_load.merge(daily_temp, on="date", how="left")

ind_rto_load["temp_f"] = ind_rto_load["temperature_2m"] * 9/5 + 32

base_temp = 65

ind_rto_load["HDD"] = np.maximum(base_temp - ind_rto_load["temp_f"], 0)
ind_rto_load["CDD"] = np.maximum(ind_rto_load["temp_f"] - base_temp, 0)

ind_rto_load["wind_chill"] = (
    35.74
    + 0.6215 * ind_rto_load["temp_f"]
    - 35.75 * (ind_rto_load["wind_speed_10m"] ** 0.16)
    + 0.4275 * ind_rto_load["temp_f"] * (ind_rto_load["wind_speed_10m"] ** 0.16)
)

RH = ind_rto_load["relative_humidity_2m"]
T = ind_rto_load["temp_f"]

ind_rto_load["heat_index_f"] = (
    -42.379
    + 2.04901523 * T
    + 10.14333127 * RH
    - 0.22475541 * T * RH
    - 0.00683783 * T**2
    - 0.05481717 * RH**2
    + 0.00122874 * T**2 * RH
    + 0.00085282 * T * RH**2
    - 0.00000199 * T**2 * RH**2
)

ind_rto_load["feels_like_temp"] = np.where(
    (ind_rto_load["temp_f"] <= 50) & (ind_rto_load["wind_speed_10m"] > 3),
    ind_rto_load["wind_chill"],
    np.where(
        (ind_rto_load["temp_f"] >= 80) & (RH >= 40),
        ind_rto_load["heat_index_f"],
        ind_rto_load["temp_f"]
    )
)

ind_rto_load["wind_dir_10m_sin"] = np.sin(np.deg2rad(ind_rto_load["wind_direction_10m"]))
ind_rto_load["wind_dir_10m_cos"] = np.cos(np.deg2rad(ind_rto_load["wind_direction_10m"]))

ind_rto_load["HDD_wind"] = ind_rto_load["HDD"] * ind_rto_load["wind_speed_10m"]
ind_rto_load["CDD_cloud"] = ind_rto_load["CDD"] * (1 - ind_rto_load["cloud_cover"] / 100)


# hour-to-hour absolute temperature change
ind_rto_load["temp_diff_1h_abs"] = ind_rto_load["temperature_2m"].diff().abs()

# cumulative total variation over past 6 hours
ind_rto_load["temp_total_variation_6h"] = (
    ind_rto_load["temp_diff_1h_abs"].rolling(6).sum()
)

# cumulative total variation over past 12 hours
ind_rto_load["temp_total_variation_12h"] = (
    ind_rto_load["temp_diff_1h_abs"].rolling(12).sum()
)

# cumulative total variation over past 24 hours
ind_rto_load["temp_total_variation_24h"] = (
    ind_rto_load["temp_diff_1h_abs"].rolling(24).sum()
)

ind_rto_load.drop(columns=["wind_chill", "temp_diff_1h_abs", "heat_index_f"], inplace=True)

# if target hour < 10, use 1 day back
# if target hour >= 10, use 2 days back
ind_rto_load["day_shift"] = np.where(ind_rto_load["hour"].isin(range(1, 10)), 1, 2)

# lookup index from historical MW
hist = (
    ind_rto_load[["datetime_ending_ept", "ind_MW"]]
    .drop_duplicates("datetime_ending_ept")
    .sort_values("datetime_ending_ept")
    .set_index("datetime_ending_ept")
)

# choose lag hours to create
lag_hours = [1, 2, 3, 6, 12, 24, 36, 48, 72, 96, 120, 144, 168]

# build lag lookup timestamps + features
helper_cols = []
feature_cols = []

for lag in lag_hours:
    time_col = f"lag{lag}_time"
    feat_col = f"MW_lag_{lag}"

    ind_rto_load[time_col] = (
        ind_rto_load["datetime_ending_ept"]
        - pd.to_timedelta(ind_rto_load["day_shift"], unit="D")
        - pd.Timedelta(hours=lag)
    )

    ind_rto_load[feat_col] = hist["ind_MW"].reindex(ind_rto_load[time_col]).to_numpy()

    helper_cols.append(time_col)
    feature_cols.append(feat_col)

# rolling stats based only on historical series
hist["MW_roll_24_base"] = hist["ind_MW"].shift(1).rolling(24).mean()
hist["MW_roll_48_base"] = hist["ind_MW"].shift(1).rolling(48).mean()
hist["MW_roll_72_base"] = hist["ind_MW"].shift(1).rolling(72).mean()
hist["MW_roll_168_base"] = hist["ind_MW"].shift(1).rolling(168).mean()

hist["MW_std_24_base"] = hist["ind_MW"].shift(1).rolling(24).std()
hist["MW_std_168_base"] = hist["ind_MW"].shift(1).rolling(168).std()

# rolling lookup time
ind_rto_load["roll_time"] = (
    ind_rto_load["datetime_ending_ept"]
    - pd.to_timedelta(ind_rto_load["day_shift"], unit="D")
)

ind_rto_load["MW_roll_24"] = hist["MW_roll_24_base"].reindex(ind_rto_load["roll_time"]).to_numpy()
ind_rto_load["MW_roll_48"] = hist["MW_roll_48_base"].reindex(ind_rto_load["roll_time"]).to_numpy()
ind_rto_load["MW_roll_72"] = hist["MW_roll_72_base"].reindex(ind_rto_load["roll_time"]).to_numpy()
ind_rto_load["MW_roll_168"] = hist["MW_roll_168_base"].reindex(ind_rto_load["roll_time"]).to_numpy()

ind_rto_load["MW_std_24"] = hist["MW_std_24_base"].reindex(ind_rto_load["roll_time"]).to_numpy()
ind_rto_load["MW_std_168"] = hist["MW_std_168_base"].reindex(ind_rto_load["roll_time"]).to_numpy()

feature_cols += [
    "MW_roll_24", "MW_roll_48", "MW_roll_72", "MW_roll_168",
    "MW_std_24", "MW_std_168"
]

# drop rows without enough history
ind_rto_load = ind_rto_load.dropna(subset=feature_cols).copy()

# optional: remove helper columns
ind_rto_load.drop(columns=["day_shift", "roll_time"] + helper_cols, inplace=True)

In [ ]:
ind_rto_load.tail()

,datetime_ending_ept,zone,ind_MW,temperature_2m,apparent_temperature,dew_point_2m,relative_humidity_2m,precipitation,rain,snowfall,...,MW_lag_144,MW_lag_168,MW_roll_24,MW_roll_48,MW_roll_72,MW_roll_168,MW_std_24,MW_std_168,temp_min_y,temp_max_y
28397,2026-05-04 21:00:00,NaN,NaN,15.9250,13.8750,9.325,74.895,0.0000,0.0,0.0,...,17559.936448,17447.372394,19409.430868,19895.667450,20075.707946,20080.415184,1910.604687,2229.676646,8.9000,20.4250
28398,2026-05-04 22:00:00,NaN,NaN,15.0340,13.2125,9.385,76.380,0.0000,0.0,0.0,...,18042.169226,17843.176801,19379.896474,19873.068025,20058.326044,20078.946874,1940.050229,2231.501443,8.9000,20.4250
28399,2026-05-04 23:00:00,NaN,NaN,15.1000,12.9000,9.405,72.910,0.0000,0.0,0.0,...,18305.352264,18207.971487,19351.557389,19853.383553,20045.083559,20079.149412,1957.477282,2231.298837,8.9000,20.4250
28400,2026-05-05 00:00:00,NaN,NaN,14.6625,12.5500,9.625,73.750,0.0125,0.0,0.0,...,18511.027252,18503.318657,19329.005764,19840.753579,20037.995871,20080.739710,1964.611407,2230.052081,14.6625,19.8225
28401,2026-05-05 10:00:00,NaN,NaN,19.8225,17.3000,10.385,57.750,0.0000,0.0,0.0,...,23130.657129,21509.736660,19347.976056,19651.555696,19973.035034,20144.425610,1954.202628,2198.259674,14.6625,19.8225


In [ ]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

results = []
feat_imp_results = []

start_date = (pd.Timestamp.today() - pd.Timedelta(days=offset)).normalize() # pd.Timestamp("2026-03-25")
end_date   = start_date + pd.Timedelta(days=2) - pd.Timedelta(hours=1) # pd.Timestamp("2026-03-26")


current_date = start_date

while current_date <= end_date:

    print(f"Running for {current_date.date()}")

    # 1. Define cutoff (day before 10:00)
    cutoff = (current_date - pd.Timedelta(days=1)).replace(hour=10)

    # 2. Train set
    train_df = ind_rto_load[
        ind_rto_load["datetime_ending_ept"] <= cutoff
    ].copy()

    # 3. Test set (full target day)
    test_start = current_date.replace(hour=1)
    test_end   = current_date.replace(hour=23) + pd.Timedelta(hours=1)

    test_df = ind_rto_load[
        (ind_rto_load["datetime_ending_ept"] >= test_start) &
        (ind_rto_load["datetime_ending_ept"] <= test_end)
    ].copy()

    # 4. Features
    target = "ind_MW"

    drop_cols = ["ind_MW", "datetime_ending_ept", "zone", "date"]
    features = [col for col in train_df.columns if col not in drop_cols]

    # 5. Time-based validation split from training data
    # use the most recent 14 days of hourly data as validation
    valid_hours = 24 * 14
    
    train_part = train_df.iloc[:-valid_hours].copy()
    valid_part = train_df.iloc[-valid_hours:].copy()

    X_train = train_part[features]
    y_train = train_part[target]

    X_valid = valid_part[features]
    y_valid = valid_part[target]

    X_test = test_df[features]

    # 6. Train model with recommended params + early stopping
    model = LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=64,
        max_depth=-1,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="rmse",
        callbacks=[
            __import__("lightgbm").early_stopping(100),
            __import__("lightgbm").log_evaluation(100)
        ]
    )

    # ---- feature importance for this day ----
    feat_imp = pd.DataFrame({
        "date": current_date,
        "feature": X_train.columns,
        "importance": model.feature_importances_
    }).sort_values(by="importance", ascending=False)

    feat_imp["rank"] = range(1, len(feat_imp) + 1)
    feat_imp_results.append(feat_imp)

    # 6. Predict
    test_df["ind_MW_pred"] = model.predict(X_test)

    # Merge with PJM forecast
    forecast_df = rto_load_forecast[
        (rto_load_forecast["datetime_ending_ept"] >= test_start) &
        (rto_load_forecast["datetime_ending_ept"] <= test_end)
    ].copy()

    compare_df = pd.merge(
        test_df,
        forecast_df[["datetime_ending_ept", "ind_forecast_load_mw"]],
        on="datetime_ending_ept",
        how="inner"
    )

    # Metrics
    mae = mean_absolute_error(compare_df["ind_forecast_load_mw"], compare_df["ind_MW_pred"])
    rmse = np.sqrt(mean_squared_error(compare_df["ind_forecast_load_mw"], compare_df["ind_MW_pred"]))

    print(f"MAE: {mae:.2f}, RMSE: {rmse:.2f}")

    compare_df["date"] = current_date
    compare_df["MAE"] = mae
    compare_df["RMSE"] = rmse

    results.append(compare_df)

    # move to next day
    current_date += pd.Timedelta(days=1)

# combine all days
ind_final = pd.concat(results).reset_index(drop=True)



Running for 2026-05-03
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005609 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21325
[LightGBM] [Info] Number of data points in the train set: 26959, number of used features: 97
[LightGBM] [Info] Start training from score 22900.870154
Training until validation scores don't improve for 100 rounds
[100]	valid_0's rmse: 792.586	valid_0's l2: 628192
[200]	valid_0's rmse: 695.236	valid_0's l2: 483353
[300]	valid_0's rmse: 679.787	valid_0's l2: 462111
[400]	valid_0's rmse: 670.47	valid_0's l2: 449530
[500]	valid_0's rmse: 664.733	valid_0's l2: 441870
[600]	valid_0's rmse: 659.715	valid_0's l2: 435224
[700]	valid_0's rmse: 658.203	valid_0's l2: 433232
[800]	valid_0's rmse: 654.265	valid_0's l2: 428063
[900]	valid_0's rmse: 650.879	valid_0's l2: 423644
[1000]	valid_0's rmse: 649.351	valid_0's l2: 421657
Did not meet early stopping. Best iteration is:
[979]	vali

In [ ]:
mean_mae = ind_final["MAE"].mean()
mean_rmse = ind_final["RMSE"].mean()
print(mean_mae, mean_rmse)

430.8813671526382 502.7068858341523


In [ ]:
import plotly.graph_objects as go

plot_df = ind_final.sort_values("datetime_ending_ept")

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=plot_df["datetime_ending_ept"],
    y=plot_df["ind_MW_pred"],
    mode="lines",
    name="Model Prediction (MW_pred)"
))

fig.add_trace(go.Scatter(
    x=plot_df["datetime_ending_ept"],
    y=plot_df["ind_forecast_load_mw"],
    mode="lines",
    name="PJM Forecast (forecast_load_mw)"
))

fig.update_layout(
    title=(
        f"Model vs PJM Forecast Load "
        f"({start_date.strftime('%Y-%m-%d')} - {end_date.strftime('%Y-%m-%d')})<br>"
        f"Model → MAE: {mean_mae:.2f}, RMSE: {mean_rmse:.2f} "
    ),
    xaxis_title="Time",
    yaxis_title="MW",
    template="plotly_white"
)

fig.show()

## Sector Union

In [ ]:
rto_load_forecast.rename(columns={"forecast_datetime_ending_ept": "datetime_ending_ept"}, inplace=True)

In [ ]:
# merge predictions together
pred_df = (
    res_final[["datetime_ending_ept" , "res_MW_pred"]]
    .merge(
        com_final[["datetime_ending_ept", "com_MW_pred"]],
        on="datetime_ending_ept",
        how="outer"
    )
    .merge(
        ind_final[["datetime_ending_ept", "ind_MW_pred"]],
        on="datetime_ending_ept",
        how="outer"
    )
)

pred_df["total_MW_pred"] = (
    pred_df["res_MW_pred"]
    + pred_df["com_MW_pred"]
    + pred_df["ind_MW_pred"]
)

pred_df = (
    pred_df
    .merge(
        rto_load[["datetime_ending_ept", "MW"]],
        on="datetime_ending_ept",
        how="left"
    )
)

pred_df = (
    pred_df
    .merge(
        rto_load_forecast[["datetime_ending_ept", "forecast_load_mw"]],
        on="datetime_ending_ept",
        how="left"
    )
)

In [ ]:
pred_df.tail()

,datetime_ending_ept,res_MW_pred,com_MW_pred,ind_MW_pred,total_MW_pred,MW,forecast_load_mw
43,2026-05-04 20:00:00,39444.724989,31502.332853,17841.553290,88788.611132,NaN,89249.0
44,2026-05-04 21:00:00,39152.730194,31832.153427,18385.535221,89370.418842,NaN,90331.0
45,2026-05-04 22:00:00,36892.225475,31565.449592,18561.398852,87019.073920,NaN,88044.0
46,2026-05-04 23:00:00,32516.475865,31108.164311,18689.351670,82313.991847,NaN,83130.0
47,2026-05-05 00:00:00,28580.253515,30017.746711,18738.826638,77336.826864,NaN,78259.0


In [ ]:
# extract date
pred_df["date"] = pred_df["datetime_ending_ept"].dt.date
pred_df['hour'] = pred_df["datetime_ending_ept"].dt.hour

daily_metrics = pred_df.groupby("date").apply(
    lambda df: pd.Series({
        "mae": mean_absolute_error(df["forecast_load_mw"], df["total_MW_pred"]),
        "rmse": np.sqrt(mean_squared_error(df["forecast_load_mw"], df["total_MW_pred"]))
    })
).reset_index()

# overall mean of daily metrics
mean_mae = daily_metrics["mae"].mean()
mean_rmse = daily_metrics["rmse"].mean()


In [ ]:
import plotly.graph_objects as go

plot_df = pred_df.sort_values("datetime_ending_ept")

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=plot_df["datetime_ending_ept"],
    y=plot_df["total_MW_pred"],
    mode="lines",
    name="Model Prediction (MW_pred)"
))

fig.add_trace(go.Scatter(
    x=plot_df["datetime_ending_ept"],
    y=plot_df["forecast_load_mw"],
    mode="lines",
    name="PJM Forecast (forecast_load_mw)"
))


fig.update_layout(
    title=(
        f"Model vs PJM Forecast Load "
        f"({start_date.strftime('%Y-%m-%d')} - {end_date.strftime('%Y-%m-%d')})<br>"
        f"Model → MAE: {mean_mae:.2f}, RMSE: {mean_rmse:.2f} "
    ),
    xaxis_title="Time",
    yaxis_title="MW",
    template="plotly_white"
)

fig.show()

### save

In [ ]:
# today = datetime.today() - timedelta(days=offset)

# start_str = today.strftime("%m%d")                 # 0325
# end_str = (today + timedelta(days=1)).strftime("%m%d")  # 0326
# run_str = today.strftime("%y%m%d")                 # 260325

# pred_df[
#     ['datetime_ending_ept', 'date', 'hour', 'total_MW_pred', 'forecast_load_mw']
# ].to_csv(
#     f'data/prediction/RTO_sector_prediction_{start_str}_to_{end_str}_at_{run_str}.csv',
#     index=False
# )
